$\renewcommand{\fl}{\operatorname{fl}}$

<div style='background-color:#002147; padding:20px; border-radius:8px'>
<h3 style='color:white;'>Aula 1: Estratégia Numérica, Erros e Ponto Flutuante</h3>
</div>

## 1️⃣ Estratégia de Solução Numérica

Processo físico → Modelo matemático → Discretização → Solução numérica.

![image.png](figures/num_model.png)

<div style="background-color: #fff3b0; padding: 12px; border-radius: 6px;">
⚠️ <b> A essência dos métodos numéricos está na discretização do contínuo.</b> ⚠️
</div>

Podemos representar o erro total como:

$$
E_{total} = E_{modelagem} + E_{truncamento} + E_{arredondamento}
$$

### 🔎 [__Exercício 1.1__](#exercício-1.1) 
1. Explique a diferença entre erro inerente e erro de truncamento.
2. Cite um exemplo de problema que não admite solução analítica.

## Introdução

O computador é uma máquina finita, capaz de armazenar e manipular apenas um **número finito de dados**. Por isso, números que não possuem representação finita, como os **irracionais** (por exemplo, $\pi$), não podem ser armazenados exatamente. Assim, cálculos como $\sin(\pi)$ não são feitos com o valor exato de $\pi$, mas sim com **aproximações numéricas** desse número.

Na prática, essas aproximações são suficientemente precisas para a maioria das aplicações. Nesta parte do curso, estudaremos **como os computadores representam números**, qual é a **qualidade dessas aproximações** e quais **problemas podem surgir devido aos erros numéricos**, especialmente quando muitas operações são realizadas em sequência. Um ponto importante é entender **como esses erros se comportam: se se acumulam ou se podem se cancelar**.

A seguir, veremos o que é $\pi$ para o computador.

In [1]:
import math
import numpy as np

np.float64(math.pi)

3.141592653589793

Como você pode ver, o computador armazena **uma aproximação** do número $\pi$. No exemplo acima aparecem os **16 primeiros dígitos significativos**, que estão corretos. Isso ocorre devido à forma como os números são representados em **dupla precisão (float64)**.

### 💡 Observação

O valor `math.pi` já é uma **aproximação em dupla precisão**, contendo cerca de **16 dígitos significativos corretos**, que é o limite da representação `float64`.

Para trabalhar com **mais dígitos de $\pi$**, é necessário utilizar **tipos numéricos de precisão arbitrária**. Em Python, isso pode ser feito com o módulo `decimal`, que permite definir explicitamente o número de dígitos usados nos cálculos.

In [2]:
from decimal import Decimal, getcontext

getcontext().prec = 50

pi = Decimal("3.14159265358979323846264338327950288419716939937510")
print(pi)

3.14159265358979323846264338327950288419716939937510


## 2️⃣ Medidas de Erro
<div style="background-color: #fff3b0; padding: 12px; border-radius: 6px;">

⚠️ Cálculo Numérico é a arte de aproximar quantidades, normalmente a solução de algum problema. Nada mais importante que saber quantificar se a aproximação é boa ou não. Nesta aula vamos ver como medir erros e qual o impacto de fazer contas em um computador.
<br></div>


Acima falamos que o computador armazena aproximações dos números (pelo menos no caso destes não admitirem representação finita). Ao se fazer uma aproximação, cometemos um pequeno *erro*. Vamos definir formalmente esse conceito.

### 📌Definições de erro
Seja $\hat{x}$ um valor que desejamos representar (ou calcular) e $x$ uma aproximação de $\hat{x}$. 

### 🔹 Erro Absoluto
O erro absoluto de $x$ com respeito a $\hat{x}$ é
$$
E_{abs}(\hat{x}) = |x - \hat{x}|.
$$
### 🔹 Erro Relativo
Já o *erro relativo* é
$$
E_{rel}(\hat{x}) = \frac{|x - \hat{x}|}{|\hat{x}|}.
$$

Observe que, para definir o erro relativo, precisamos que $\hat{x} \neq 0$.

### 🔹 Erro adimensional

Um outro tipo de erro que pode ser útil é o erro adimensional definido por
$$
E_{adm}(\hat{x}) = \frac{|x - \hat{x}|}{L},
$$
em que $L$ é uma constante que representa de alguma forma valores típicos esperados.

Vejamos agora dois exemplos. Considere que obtivemos $x = 0{,}9273$ para aproximar $\hat{x} = 1$. Quais os erros associados? Temos
$$
E_{abs} = |0{,}9273 - 1{,}0| = 0{,}0727,\quad\quad E_{rel} = \frac{|0{,}9273 - 1{,}0|}{|1{,}0|} = 0{,}0727.
$$
Nesse caso, como o valor desejado tem módulo 1, os erros absoluto e relativo coincidem.

Já para $x = 0{,}9273$ para aproximar $0{,}9$, teríamos
$$
E_{abs} = |0{,}9273 - 0{,}9| = 0{,}0273,\quad\quad E_{rel} = \frac{0{,}0273}{0{,}9} = 0{,}0303333333\ldots.
$$
Aqui o erro relativo é maior que o absoluto, dando mais peso ao erro porque o número que desejávamos aproximar tem módulo menor do que 1.

## 3️⃣ Representação de números no computador (Sistema de Ponto Flutuante - SPF)

<div class="alert alert-info">

Seria ideal que todas as contas feitas no computador fossem **exatas**, mas isso nem sempre é possível. Para entender por quê, precisamos compreender **como os números são representados internamente** e qual é a qualidade dessa representação.

</div>

Um computador ou calculadora representa um número real no sistema denominado **aritmética de ponto flutuante**. Neste sistema, o número $\hat{x}$ será representado na forma

$$
\hat{x}=\mathrm{fl}(x)=\pm (0.d_1 d_2 \ldots d_t)\times \beta^{e}, \quad d_1\neq0
$$

onde:

- $\beta$ é a base em que a máquina opera;

- $t$ é o número de dígitos na mantissa, com

$$
0 \le d_j \le (\beta - 1), \quad j = 1,\ldots,t, \quad d_1 \ne 0;
$$

- $e$ é o expoente no intervalo

$$
e \in [L, U].
$$
**Expoente $e$**, que determina a posição da vírgula, limitado entre um valor mínimo $L$ e máximo $U$.


Assim, um sistema de ponto flutuante é denotado por

$$
\mathbb{F}(\beta,t,L,U).
$$

---

<div class="alert alert-info">

💻 Considere o sistema

$$
\mathbb{F}(10,4,-99,99),
$$

onde:

- base $\beta=10$,
- mantissa com **4 dígitos**,
- expoente entre **$-99$ e $99$**.

</div>

### Exemplo

Queremos representar o número

$$
x=0{,}034.
$$

No sistema descrito, ele é escrito como

$$
\hat{x}=\mathrm{fl}(x)=0{,}3400 \cdot 10^{-1}.
$$

---


E como seria representado o número $\pi$ em um sistema de ponto flutuante?

Primeiro, vamos relembrar o seu valor.

In [3]:
math.pi

3.141592653589793

A melhor representação que podemos obter no sistema $\mathbb{F}(10,4,-99,99)$ é

$$
\mathrm{fl}(\pi)=0{,}314\textcolor{red}{2} \times 10^1.
$$


⚠️ Note que, no sistema $\mathbb{F}(10,4,-99,99)$, os números são escritos na forma

$$
\pm (0.d_1d_2d_3d_4)\times 10^e, \quad d_1 \ne 0.
$$

O menor número, em valor absoluto, representado nesta máquina é:

- a menor mantissa normalizada $0{,}1000$,
- e o menor expoente $e=-99$:

$$
m = 0{,}1000 \times 10^{-99}.
$$

O maior número representável ocorre quando usamos

- a maior mantissa $0{,}9999$,
- e o maior expoente $e=99$:

$$
M = 0{,}9999 \times 10^{99}.
$$

---

## 🖥️ Agora, qual é o sistema de ponto flutuante adotado no computador?  
Quase todas as máquinas modernas implementam o padrão **IEEE 754**, que define como números reais são representados em ponto flutuante.

Esse padrão define dois tipos básicos de números.

### 🔹Precisão simples

Os números de **precisão simples**, geralmente representados como `float32` (por exemplo, na biblioteca **NumPy**), ocupam **32 bits.**

### 🔹 Precisão dupla

Já os números de **precisão dupla**, que correspondem ao tipo padrão `float` em Python (equivalente a `float64` em **NumPy**), ocupam **64 bits.**

Quem quiser mais informações sobre o padrão IEEE 754 pode consultar este [texto](http://steve.hollasch.net/cgindex/coding/ieeefloat.html).

### Resumo da representação IEEE 754

| Tipo | Bits totais | Sinal | Expoente | Mantissa |
|-----|-----|-----|-----|-----|
| Precisão simples | 32 | 1 | 8 | 23 |
| Precisão dupla | 64 | 1 | 11 | 52 |

Em Python:

- `float32` → precisão simples  
- `float64` → precisão dupla (padrão)

### Obtendo essas informações em Python

Em Python, essas informações podem ser obtidas usando a biblioteca **NumPy**:

```python
import numpy as np

np.finfo(np.float64)   # informações sobre precisão dupla
np.finfo(np.float32)   # informações sobre precisão simples

```


In [4]:
import numpy as np

np.finfo(np.float32)   # informações sobre precisão simples

finfo(resolution=1e-06, min=-3.4028235e+38, max=3.4028235e+38, dtype=float32)

In [5]:
np.finfo(np.float64)   # informações sobre precisão simples

finfo(resolution=1e-15, min=-1.7976931348623157e+308, max=1.7976931348623157e+308, dtype=float64)

📝[__Exercício 1.2__](#Exercicio-1.2) Quais são os valores mais distantes e mais próximos de zero para presição simples e dupla? 



<div class="alert alert-info">

### Menor número representável em precisão dupla

Além das informações sobre **precisão**, **expoentes** e **limites do sistema**, também é interessante observar qual é o **menor número positivo representável** pelo computador.

Em precisão dupla (`float64`), o menor número **normalizado positivo** é aproximadamente

$$
2.2250738585072014 \times 10^{-308}.
$$

Esse valor pode ser obtido em Python usando:

```python
np.finfo(np.float64).tiny

In [6]:
np.finfo(np.float64).tiny

2.2250738585072014e-308

In [7]:
np.finfo(np.float32).tiny

1.1754944e-38

<div class="alert alert-info">

### Números subnormais

O padrão **IEEE 754** também permite representar números ainda menores chamados **números subnormais** (ou *denormalized numbers*).

Esses números são utilizados para preencher o intervalo entre o menor número normalizado e o zero, permitindo representar valores **mais próximos de zero**.

O menor número positivo diferente de zero representável em precisão dupla é aproximadamente

$$
4.94 \times 10^{-324}.
$$

</div>

<div class="alert alert-info">

### Obtendo o menor número positivo em Python

Podemos obter o menor número positivo representável usando a função `nextafter` da biblioteca **NumPy**.

```python
import numpy as np

np.nextafter(0,1)

In [8]:
np.nextafter(0,1)

5e-324

---
<div style="background-color: #fff3b0; padding: 12px; border-radius: 6px;">
⚠️ <b>Valores acima ou abaixo da capacidade de representação pode causar<span style="color:blue"> overﬂow </span> e <span style="color:red">underﬂow</span> respectivamente.</b>
</div>


📝[__Exemplo 1.1__](#exemplo-1.1)

```python
# overflow
x = 1e308
print(x*1e10)   # inf

# underflow
y = 5e-324
print(y/1e10)   # 0.0
```

In [9]:
x = 1e308
print(x*1e10) 

inf


In [10]:
y = 5e-324
print(y/1e10)   # 0.0

0.0


---
[__Exercício 01.01.1__](#exercicio-01.01.1):
Os "números" $ +\infty $, $ -\infty $ e NaN podem ser acessados através de `float('inf')`, `float('-inf')` e `float('NaN')`, respectivamente. Quanto é $ (+\infty) \cdot 0 $?

<div class="alert alert-info">

### Observação importante

Esse exemplo mostra uma característica fundamental da aritmética de ponto flutuante:

- o conjunto de números representáveis pelo computador é **finito**
- existem **lacunas entre números representáveis**
- números muito pequenos podem ser **arredondados para zero**

Essas limitações são responsáveis por muitos **erros numéricos em cálculos computacionais**.

</div>

<div class="alert alert-info">

### Epsilon da máquina

Uma consequência importante da representação em ponto flutuante é que **nem todos os números reais podem ser representados exatamente**.

Chamamos de **epsilon da máquina**, denotado por $\epsilon_{mach}$, o **menor número positivo $\epsilon$ tal que**

$$
\mathrm{fl}(1+\epsilon) > 1,
$$

onde $\mathrm{fl}(\cdot)$ representa o resultado obtido pelo computador após arredondamento.

Esse valor fornece uma medida da **precisão numérica do sistema de ponto flutuante**.

</div>

<div class="alert alert-info">

### Valor do epsilon da máquina

No padrão **IEEE 754**, temos aproximadamente:

| Precisão | Epsilon da máquina |
|---|---|
| precisão simples (`float32`) | $1.19\times10^{-7}$ |
| precisão dupla (`float64`) | $2.22\times10^{-16}$ |

Esses valores podem ser obtidos em Python usando a biblioteca **NumPy**.

</div>

<div class="alert alert-info">

### Obtendo o epsilon da máquina em Python

```python
import numpy as np

np.finfo(np.float32).eps
np.finfo(np.float64).eps

In [11]:
import numpy as np 

np.finfo(np.float32).eps # epsilon da máquina em precisão simples

1.1920929e-07

In [12]:
import numpy as np

np.finfo(np.float64).eps # epsilon da máquina em precisão dupla

2.220446049250313e-16

<div class="alert alert-info">

### Experimento numérico

Vamos verificar o comportamento do epsilon da máquina na prática.

```python
import numpy as np

eps = np.finfo(np.float64).eps

print(1 + eps)
print(1 + eps/2)

In [13]:
eps = np.finfo(np.float64).eps

print(1 + eps)
print(1 + eps/2)

1.0000000000000002
1.0


<div class="alert alert-info">

### Interpretação

Observe que:

- $1 + \epsilon > 1$
- $1 + \epsilon/2 = 1$

Isso ocorre porque **$\epsilon/2$ é pequeno demais para alterar o número 1 dentro da precisão do computador**.

Ou seja, o resultado é arredondado para o **número representável mais próximo**, que neste caso é o próprio **1**.

</div>

<div class="alert alert-info">

### Consequência importante

Esse exemplo mostra que, em aritmética de ponto flutuante,

$$
\mathrm{fl}(1 + u) = 1
$$

sempre que $u$ for **suficientemente pequeno**.

Isso significa que existem números **diferentes de zero** que, quando somados a 1 no computador, **não alteram o resultado armazenado**.

Esse fenômeno é uma das principais fontes de **erro de arredondamento em cálculos numéricos**.

</div>

<div class="alert alert-info">

### Exemplo clássico: $0.1 + 0.2$

Um exemplo muito conhecido de erro de representação em ponto flutuante é a soma

$$
0.1 + 0.2.
$$

Matematicamente esperamos obter

$$
0.3.
$$

Vamos verificar o que acontece no computador.

```python
0.1 + 0.2

In [14]:
0.1+0.2

0.30000000000000004

## Conversão da base 10 para a notação IEEE-754

Considere o número decimal

$$
0.1_{10}
$$

### Converter para binário

Multiplicando sucessivamente por 2:

| passo | valor ×2 | bit |
|------|------|------|
|1|0.1 × 2 = 0.2|0|
|2|0.2 × 2 = 0.4|0|
|3|0.4 × 2 = 0.8|0|
|4|0.8 × 2 = 1.6|1|
|5|0.6 × 2 = 1.2|1|
|6|0.2 × 2 = 0.4|0|
|...|...|...|

Assim,

$$
0.1_{10} =
0.0001100110011001100110011001100110011\ldots_2
$$

ou

$$
0.1_{10} = 0.0001\overline{1001}_2
$$

---

### Representação 

$$
0\;|\;01111111011\;|\;1001100110011001100110011001100110011001100110011010
$$

A função abaixo mostra a representação binária de um número de ponto flutuante (`float64`) no padrão **IEEE-754**, separando **sinal, expoente e mantissa**.


In [15]:
import struct

def float64_bits(x):
    """Mostra a representação binária (IEEE-754) de um número float64."""
    
    bits = struct.unpack('>Q', struct.pack('>d', x))[0]
    b = f'{bits:064b}'

    print(f"Número: {x}")
    print("Sinal | Expoente      | Mantissa")
    print(f"{b[0]}     | {b[1:12]} | {b[12:]}")

In [16]:
float64_bits(0.1)

Número: 0.1
Sinal | Expoente      | Mantissa
0     | 01111111011 | 1001100110011001100110011001100110011001100110011010



O valor armazenado é

$$
0.1 \approx 0.10000000000000000555\ldots
$$

In [17]:
format(0.1, ".20f")

'0.10000000000000000555'


---

### Representação resumida de 0.2

Representação IEEE-754 (float64):

$$
0\;|\;01111111100\;|\;1001100110011001100110011001100110011001100110011010
$$

Valor armazenado:

$$
0.2 \approx 0.20000000000000001110\ldots
$$

In [18]:
float64_bits(0.2)

Número: 0.2
Sinal | Expoente      | Mantissa
0     | 01111111100 | 1001100110011001100110011001100110011001100110011010


✅ Forma de visualizar a aproximação armazenada:

In [19]:
format(0.2, ".20f")

'0.20000000000000001110'

---

### Consequência em Python

```python
0.1 + 0.2

In [20]:
0.1 + 0.2

0.30000000000000004

## Exemplo: Somatórios e erro numérico

Considere o seguinte somatório:

$$
S = \sum_{i=1}^{30000} a
$$

onde $a$ é um número real constante.

Matematicamente, o resultado exato é

$$
S = 30000 \cdot a
$$

Vamos calcular esse somatório no computador utilizando Python para dois valores de $a$:

- $a = 0.5$
- $a = 0.11$

O objetivo é observar que **nem todos os números decimais são representados exatamente no computador**, o que pode gerar **pequenos erros acumulados** em somatórios com muitas operações.

### Somatório com $a = 0.5$

Neste caso calculamos

$$
S = \sum_{i=1}^{30000} 0.5
$$

O valor exato deveria ser

$$
S = 30000 \times 0.5 = 15000
$$

Vamos verificar o resultado obtido pelo computador.

In [21]:
# Inicializando o somatório
total = 0.0

# Realizando o somatório de i=1 até i=30000 com incrementos de 0.5
i = 1.0
while i <= 30000:
    total += 0.5
    i += 1

# Exibindo o resultado
print(f"O somatório de 0.5 de i=1 até i=30000 é: {total:.5f}")

O somatório de 0.5 de i=1 até i=30000 é: 15000.00000


Neste caso o resultado obtido é **exatamente 15000**.

Isso ocorre porque o número **0.5 possui representação exata em binário**, sendo uma fração do tipo:

$$
0.5 = \frac{1}{2}
$$

Portanto, o computador consegue representá-lo **sem erro**.

### Somatório com $a = 0.11$

Agora calculamos

$$
S = \sum_{i=1}^{30000} 0.11
$$

O valor exato deveria ser

$$
S = 30000 \times 0.11 = 3300
$$

Vamos verificar o resultado computacional.

In [22]:
# Inicializando o somatório
total = 0.0

# Realizando o somatório
i = 1.0
while i <= 30000:
    total += 0.11
    i += 1

# Resultado
print("Resultado do somatório:", total)

Resultado do somatório: 3300.0000000006285


Neste caso o resultado obtido **não será exatamente 3300**.

Isso ocorre porque $0.11$ **não possui representação exata em binário**.  
Assim, o computador armazena uma **aproximação** desse número.

Quando realizamos milhares de somas sucessivas, esse pequeno erro de representação **se acumula**, gerando uma pequena diferença no resultado final.

Esse fenômeno é conhecido como **erro de arredondamento em ponto flutuante**.

---

> 📚 **Referência teórica**
>
>> **Material:** *Sistemas de Ponto Flutuante*  
> https://www.ime.unicamp.br/~biloti/an/211/pf-02.html#aprofunde
>
>> Paulo J. S. Silva  
> https://github.com/pjssilva/ms211
>
>> **Livro:** *Cálculo numérico: aspectos teóricos e computacionais*  
> Márcia A. Gomes Ruggiero; Vera Lúcia da Rocha Lopes  
> São Paulo: Makron Books, 1996.